# **Universal Notebook Environment Setup**

In [1]:
import os
import sys
import wandb

# --- AUTOMATIC ENVIRONMENT SETUP ---
KAGGLE_RUN = os.path.exists('/kaggle/working')

if KAGGLE_RUN:
    print("Running on Kaggle. Setting up paths...")
    # Kaggle Secret for W&B
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

    # Create a symlink so /content/artifacts works on Kaggle
    os.makedirs('/content', exist_ok=True)
    if not os.path.exists('/content/artifacts'):
        # Map Kaggle input artifacts to Colab path
        os.system('ln -s /kaggle/input /content/artifacts')
else:
    print("Running on Google Colab.")
    # Colab Secret for W&B
    try:
        from google.colab import userdata
        wandb_api_key = userdata.get('WANDB_API_KEY')
        wandb.login(key=wandb_api_key)
    except Exception as e:
        print("W&B Secret not found, you may need to login manually.")
        wandb.login()

Running on Kaggle. Setting up paths...


# **Fetch Augmented Images and Model From W&B**

In [2]:
import os
import sys
import wandb

# Initialize a single W&B run for environment setup
run = wandb.init(project="pcb-defect-detection", job_type="setup")

print("--- Downloading Dataset Artifact ---")
# 1. Download Dataset
artifact_dataset = run.use_artifact('pcb-augmented-dataset:latest', type='dataset')
dataset_dir = artifact_dataset.download()
print(f"Dataset ready at: {os.path.abspath(dataset_dir)}")

print("\n--- Downloading Core Model Artifact ---")
# 2. Download Core Model Code
artifact_code = run.use_artifact('pcb-core-models:latest', type='model')
model_dir = artifact_code.download()

# Add to sys.path to allow immediate import
if model_dir not in sys.path:
    sys.path.insert(0, model_dir)
print(f"Core model ready at: {model_dir}")

run.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: naufalsatya (nsp-deep-learning-projects) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


--- Downloading Dataset Artifact ---


wandb: Downloading large artifact 'pcb-augmented-dataset:latest', 3649.25MB. 40306 files...
wandb:   40306 of 40306 files downloaded.  
Done. 00:03:25.2 (17.8MB/s)


Dataset ready at: /kaggle/working/artifacts/pcb-augmented-dataset:v4

--- Downloading Core Model Artifact ---


wandb:   2 of 2 files downloaded.  


Core model ready at: /kaggle/working/artifacts/pcb-core-models:v3


# **Device Setup & W&B Tracking Initialization**

In [ ]:
import os
import cv2
import torch
import wandb

# Initialize the experiment tracking run
run = wandb.init(
    project="pcb-defect-detection",
    name="Model_Architecture_Training",
    notes="Training the MobileViT + LeYOLO architecture and metric logging with validation",
    config = {
        "epochs": 50,
        "batch_size": 16, # Can't be more than this
        "image_size": 640,
        "loss": "v8DetectionLoss",
        "optimizer": "AdamW,lr=1e-3,weight_decay=1e-4",
        "scheduler": "CosineAnnealingLR,eta_min=1e-6",
        "conf_threshold": 0.05,
        "iou_threshold": 0.45,
        "train/val_split": "80/20",
        "backbone": "MobileViT-XXS",
        # Using default values
        "multiplier": 1.5,
        "neck_depth": 2,
    },
)
config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# **Dataset Loader From W&B**

In [4]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader

class PCBDataset(Dataset):
    def __init__(self, img_dir, label_dir, img_size=config.image_size):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        self.img_size  = img_size
        self.img_names = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        # Load and preprocess image
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))

        # Normalize and convert to tensor (CHW format)
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0

        # Load YOLO format labels
        label_path = os.path.join(self.label_dir, self.img_names[idx].replace('.jpg', '.txt'))
        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        labels.append(int(float(parts[0])))  # handles '5.0' format
                        boxes.append([float(x) for x in parts[1:]])

        targets = {
            "boxes":   torch.tensor(boxes,  dtype=torch.float32),
            "labels":  torch.tensor(labels, dtype=torch.int64),
            "raw_img": img  # kept for W&B visualization
        }
        return img_tensor, targets


def collate_fn(batch):
    images   = torch.stack([item[0] for item in batch])
    raw_imgs = [item[1]["raw_img"] for item in batch]

    batch_idx_list, cls_list, box_list = [], [], []

    for b_idx, item in enumerate(batch):
        targets   = item[1]
        num_boxes = len(targets["labels"])
        if num_boxes > 0:
            batch_idx_list.append(torch.full((num_boxes,), b_idx, dtype=torch.long))
            cls_list.append(targets["labels"].unsqueeze(1))
            box_list.append(targets["boxes"])

    if len(batch_idx_list) > 0:
        batch_dict = {
            'batch_idx': torch.cat(batch_idx_list, dim=0),
            'cls':       torch.cat(cls_list,       dim=0),
            'bboxes':    torch.cat(box_list,        dim=0)
        }
    else:
        batch_dict = {
            'batch_idx': torch.empty(0,       dtype=torch.long),
            'cls':       torch.empty((0, 1),  dtype=torch.long),
            'bboxes':    torch.empty((0, 4),  dtype=torch.float32)
        }

    return images, batch_dict, raw_imgs


# ── DATASET & DATALOADER SETUP ─────────────────────────────────────────────────
DATA_DIR     = os.path.join(dataset_dir, 'train')
full_dataset = PCBDataset(
    os.path.join(DATA_DIR, "images"),
    os.path.join(DATA_DIR, "labels")
)

# 80/20 train/val split — fixed seed for reproducibility
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size   = total_size - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# Calculate optimal workers for asynchronous loading
workers = min(os.cpu_count() or 2, 4)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=workers,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=workers,
    pin_memory=True
)

print(f"Dataset split — Train: {train_size} | Val: {val_size}")
print(f"Dataloader workers configured: {workers}")

Dataset split — Train: 16122 | Val: 4031
Dataloader workers configured: 4


# **Architecture Module Configuration**

In [5]:
%pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
import importlib.util
import os

def load_module(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Unable to load module {module_name} from {file_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

model_mobilevit_path = os.path.join(model_dir, "mobilevit_xxs.py")
model_leyolo_path = os.path.join(model_dir, "leyolo_head.py")

mobilevit_xxs = load_module("mobilevit_xxs", model_mobilevit_path)
leyolo_head = load_module("leyolo_head", model_leyolo_path)

print("Model ready to be used")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model ready to be used


# **Tensor Output Matching for Double Check**

In [7]:
from leyolo_head import LeYOLO

# LeYOLO + MobileViT-XXS
model = LeYOLO(num_classes=6, multiplier=config.multiplier, depth=2, image_size=config.image_size).to(device)

model.train()
img_dummy = torch.zeros(1, 3, 640, 640).to(device)
with torch.no_grad():
    out = model(img_dummy)

print(f"Output scales: {len(out)}")
print(f"Head strides : {model.head.stride}")
for i, o in enumerate(out):
    print(f"  Scale {i}: {o.shape}")

Output scales: 3
Head strides : tensor([ 8., 16., 32.])
  Scale 0: torch.Size([1, 70, 80, 80])
  Scale 1: torch.Size([1, 70, 40, 40])
  Scale 2: torch.Size([1, 70, 20, 20])


# **Native LeYOLO + MobileViT Model Architecture**

In [8]:
import torch
from leyolo_head import LeYOLO

# Configure parameters from W&B config
neck_multiplier = getattr(config, "multiplier", 1.5)
neck_depth = getattr(config, "neck_depth", 2)

# Load the original LeYOLO directly from the script (untuned)
model = LeYOLO(
    num_classes=6,
    multiplier=neck_multiplier, 
    depth=neck_depth,
    image_size=config.image_size,
).to(device)

print("Original LeYOLO + MobileViT initialized.")

Original LeYOLO + MobileViT initialized.


# **Custom Model Evaluation Function**

In [9]:
%pip install -q torchmetrics

Note: you may need to restart the kernel to use updated packages.


In [10]:
import torch
import torchvision.ops as ops
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def evaluate_model(
    model,
    val_loader,
    device,
    conf_threshold: float = config.conf_threshold,
    iou_threshold: float = config.iou_threshold,
) -> dict:

    model.eval()

    # Local instance — no global state pollution
    metric = MeanAveragePrecision(
        box_format='xyxy',
        iou_type='bbox',
        class_metrics=True,
        max_detection_thresholds=[1, 10, 50]
    )

    with torch.no_grad():
        for images, target_dict, _ in val_loader:
            images = images.to(device)

            # Dynamic sizing — works for any resolution
            _, _, H, W = images.shape

            decoded_preds = model(images)  # [B, 4+NC, Anchors]

            preds_list, targets_list = [], []

            for b in range(images.shape[0]):
                # ── GROUND TRUTH ──────────────────────────────────────────
                gt_mask = target_dict['batch_idx'] == b
                gt_boxes_norm = target_dict['bboxes'][gt_mask]  # normalized cxcywh
                # Explicit dtype — avoids silent mismatch in torchmetrics
                gt_labels = target_dict['cls'][gt_mask].squeeze(-1).to(torch.int64)

                if len(gt_boxes_norm) > 0:
                    x_c, y_c, bw, bh = gt_boxes_norm.unbind(1)
                    gt_boxes_xyxy = torch.stack([
                        (x_c - bw / 2) * W, (y_c - bh / 2) * H,
                        (x_c + bw / 2) * W, (y_c + bh / 2) * H,
                    ], dim=1).to(device)
                else:
                    gt_boxes_xyxy = torch.empty((0, 4), device=device)

                targets_list.append({
                    "boxes":  gt_boxes_xyxy,
                    "labels": gt_labels.to(device),
                })

                # ── PREDICTIONS + NMS ──────────────────────────────────────
                preds       = decoded_preds[b]
                pred_boxes  = preds[:4, :].T
                pred_scores = preds[4:, :].T

                # ✅ YOLOv8 head already applies sigmoid in eval mode! Do not double sigmoid.
                max_scores, class_indices = pred_scores.max(dim=1)
                conf_mask = max_scores > conf_threshold

                f_boxes  = pred_boxes[conf_mask]
                f_scores = max_scores[conf_mask]
                f_labels = class_indices[conf_mask]

                if len(f_boxes) > 0:
                    x_c, y_c, bw, bh = f_boxes.unbind(1)
                    f_boxes_xyxy = torch.stack([
                        (x_c - bw / 2), (y_c - bh / 2),
                        (x_c + bw / 2), (y_c + bh / 2),
                    ], dim=1)

                    keep = ops.nms(f_boxes_xyxy, f_scores, iou_threshold=iou_threshold)

                    preds_list.append({
                        "boxes":  f_boxes_xyxy[keep],
                        "scores": f_scores[keep],
                        "labels": f_labels[keep].to(torch.int64),
                    })
                else:
                    preds_list.append({
                        "boxes":  torch.empty((0, 4),                   device=device),
                        "scores": torch.empty((0,),                     device=device),
                        "labels": torch.empty((0,), dtype=torch.int64,  device=device),
                    })
            metric.update(preds_list, targets_list)

    results = metric.compute()

    # Restore training mode before returning
    model.train()
    return results

print("Custom model evaluation initialized.")

Custom model evaluation initialized.


# **Create Visualize Predictions for Validation**

In [11]:
def visualize_predictions(model, viz_batch, device, epoch,
                           conf_threshold=config.conf_threshold,
                           iou_threshold=config.iou_threshold):

    model.eval()
    drawn_images = []
    with torch.no_grad():
        # Iterate over the pre-collected fixed evaluation batches
        for viz_images, viz_targets, viz_raw in viz_batch:
            viz_images = viz_images.to(device)
            decoded_preds = model(viz_images)

            viz_img = viz_raw[0].copy()
            h, w, _ = viz_img.shape

            # ── GROUND TRUTH (Green) ───────────────────────────────────────────
            gt_mask = viz_targets['batch_idx'] == 0
            for box in viz_targets['bboxes'][gt_mask]:
                x_c, y_c, bw, bh = box.cpu().numpy()
                x1, y1 = int((x_c - bw/2)*w), int((y_c - bh/2)*h)
                x2, y2 = int((x_c + bw/2)*w), int((y_c + bh/2)*h)
                cv2.rectangle(viz_img, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # ── PREDICTIONS (Red) with NMS ────────────────────────────────────
            preds = decoded_preds[0].cpu()
            pred_boxes  = preds[:4, :].T   # [Anchors, 4] cxcywh
            pred_scores = preds[4:, :].T   # [Anchors, NC]

            # YOLOv8 head already applies sigmoid in eval mode!
            max_scores, class_indices = pred_scores.max(dim=1)
            conf_mask = max_scores > conf_threshold

            f_boxes   = pred_boxes[conf_mask]
            f_scores  = max_scores[conf_mask]
            f_labels  = class_indices[conf_mask]

            if len(f_boxes) > 0:
                x_c, y_c, bw, bh = f_boxes.unbind(1)
                f_boxes_xyxy = torch.stack([
                    (x_c - bw/2), (y_c - bh/2),
                    (x_c + bw/2), (y_c + bh/2)
                ], dim=1)
                keep = ops.nms(f_boxes_xyxy, f_scores, iou_threshold=iou_threshold)

                for idx in keep:
                    bx1, by1, bx2, by2 = f_boxes_xyxy[idx].numpy()
                    score  = f_scores[idx].item()
                    cls_id = f_labels[idx].item()
                    cv2.rectangle(viz_img, (int(bx1), int(by1)), (int(bx2), int(by2)), (255, 0, 0), 2)
                    cv2.putText(
                        viz_img,
                        f"cls{cls_id}: {score:.2f}",
                        (int(bx1), int(by1) - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1
                    )

            drawn_images.append(viz_img)

    model.train()
    return drawn_images

print("Visual prediction initialized.")

Visual prediction initialized.


# **Overfit Loop & W&B Training Tracking**

In [13]:
import os
import torch
import torchvision.ops as ops
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from ultralytics.utils.loss import v8DetectionLoss

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── 1. DUMMY WRAPPER ───────────────────────────────────────────────────────────
class DummyModelConfig:
    def __init__(self, full_model, target_device):
        self._full_model = full_model
        self.device      = target_device
        class Args:
            box, cls, dfl, cls_pw = 7.5, 0.5, 1.5, 2.0 # Was 1
        self.args = Args()
        class MockDetectHead:
            def __init__(self, head, target_device):
                self.stride  = head.stride.to(target_device)  # ✅ ensure on correct device
                self.nc      = head.nc
                self.no      = head.no
                self.device  = target_device
                self.reg_max = head.ch
                self.use_dfl = True
        self.model = [MockDetectHead(full_model.head, target_device='cuda')]
    def parameters(self):
        return self._full_model.parameters()

# ── 2. LOSS, OPTIMIZER, SCHEDULER & SCALER ─────────────────────────────────────
dummy_config = DummyModelConfig(model, device)
yolo_loss_fn = v8DetectionLoss(dummy_config)

epochs    = config.epochs
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

scheduler = CosineAnnealingLR(optimizer, T_max=epochs - 3, eta_min=5e-5)

# Initialize AMP Scaler for Mixed Precision
scaler = GradScaler('cuda')

# ── 3. BEST MODEL TRACKING ─────────────────────────────────────────────────────
best_map50 = 0.0
best_epoch = 0
no_improve = 0

# Parameter Warmup
warmup_epochs = 3
start_lr      = 1e-6

# Run once before the training loop
VIZ_SAMPLES = 4  # one per difficulty level roughly
viz_batch = []
for i, (imgs, targets, raws) in enumerate(val_loader):
    # Make sure to move to device here since pin_memory/collate outputs to CPU now
    imgs = imgs.to(device, non_blocking=True)
    for k, v in targets.items():
        targets[k] = v.to(device, non_blocking=True)

    viz_batch.append((imgs, targets, raws))
    if i >= VIZ_SAMPLES:
        break

print(f"Launching Training Run ({epochs} epochs) with Mixed Precision & Async Loading...")

# ── 4. TRAINING LOOP ───────────────────────────────────────────────────────────
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    # 🔄 ATUR LEARNING RATE (Warmup vs Main Scheduler)
    if epoch < warmup_epochs:
        # Hitung kenaikan LR secara linear selama fase warmup
        lr_step = start_lr + (1e-3 - start_lr) * (epoch / warmup_epochs)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr_step

    current_lr = optimizer.param_groups[0]['lr']
    print(f"\n   Epoch [{epoch}/{epochs}] | Learning Rate: {current_lr:.6f}")

    for batch_idx, (images, target_dict, raw_imgs) in enumerate(train_loader):
        # Move tensors to device (async overlap with computation due to pin_memory)
        images = images.to(device, non_blocking=True)
        for k, v in target_dict.items():
            target_dict[k] = v.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True) # Slightly faster than zero_grad()

        # ✅ Run forward pass under autocast for Automatic Mixed Precision (AMP)
        with autocast('cuda'):
            # Get raw feature maps from model
            feats = model(images)

            # Build the dict v8DetectionLoss 8.4.47 expects
            box_ch = model.head.ch * 4  # 16 * 4 = 64
            nc     = model.head.nc      # 6

            boxes_list  = []
            scores_list = []
            for f in feats:
                B, C, H, W = f.shape
                f_flat = f.view(B, C, -1)
                boxes_list.append(f_flat[:, :box_ch])
                scores_list.append(f_flat[:, box_ch:])

            # Convert prediction results to FP32 before entering the Loss function
            preds = {
                "feats":  [f.float() for f in feats],
                "boxes":  torch.cat(boxes_list,  dim=2).float(),
                "scores": torch.cat(scores_list, dim=2).float(),
            }

        # Hitung Loss di luar autocast (FP32)
        loss, loss_items = yolo_loss_fn(preds, target_dict)
        loss = loss.sum()

        # Jika loss mendadak NaN karena bad batch, lewati langkah optimasi agar tidak merusak bobot
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"⚠️ [Warning] Loss NaN terdeteksi pada Epoch {epoch} Batch {batch_idx}. Melewati batch ini.")
            continue

        # ✅ Scale backward pass
        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    if epoch >= warmup_epochs:
        scheduler.step()

    avg_epoch_loss = epoch_loss / len(train_loader)
    current_lr     = scheduler.get_last_lr()[0]

    wandb.log({
        "Train_Loss":    avg_epoch_loss,
        "Learning_Rate": current_lr,
        "Epoch":         epoch
    })

    # ── VALIDATION ─────────────────────────────────────────────────────────────
    if epoch % 2 == 0 or epoch == epochs - 1:
        print(f"\nRunning Evaluation — Epoch [{epoch}/{epochs}]...")

        val_metrics = evaluate_model(
            model, val_loader, device,
            conf_threshold=config.conf_threshold,
            iou_threshold=config.iou_threshold,
        )
        # Pass the pre-collected viz_batch instead of val_loader
        viz_imgs = visualize_predictions(
            model, viz_batch, device, epoch,
            conf_threshold=config.conf_threshold,
            iou_threshold=config.iou_threshold,
        )

        current_map50 = val_metrics['map_50'].item()

        wandb.log({
            "Val_mAP_50":             current_map50,
            "Val_Recall":             val_metrics['mar_50'].item(),
            "Val_mAP_50_95":          val_metrics['map'].item(),
            # Log multiple images correctly
            "Validation/Predictions": [wandb.Image(img, caption=f"Epoch {epoch} - Sample {i+1}") for i, img in enumerate(viz_imgs)],
            "Epoch":                  epoch
        })

        print(f"Epoch [{epoch}/{epochs}] | Loss: {avg_epoch_loss:.4f} "
              f"| mAP@0.5: {current_map50:.4f} "
              f"| Recall: {val_metrics['mar_50'].item():.4f} "
              f"| LR: {current_lr:.6f}")

        if current_map50 > best_map50:
            best_map50   = current_map50
            best_epoch   = epoch
            no_improve   = 0
            torch.save(model.state_dict(), "mobilevit_leyolo_best.pt")
            print(f"  ★ New best model saved — mAP@0.5: {best_map50:.4f} at epoch {best_epoch}")
        else:
            no_improve += 1
            print(f"  No improvement for {no_improve * 10} epochs")

# ── 5. SAVE FINAL WEIGHTS ──────────────────────────────────────────────────────
torch.save(model.state_dict(), "mobilevit_leyolo_final.pt")
wandb.save("mobilevit_leyolo_best.pt")
wandb.save("mobilevit_leyolo_final.pt")
wandb.finish()

print(f"\nTraining Complete!")
print(f"  Best : mobilevit_leyolo_best.pt  (epoch {best_epoch}, mAP@0.5: {best_map50:.4f})")
print(f"  Final: mobilevit_leyolo_final.pt")

Launching Training Run (50 epochs) with Mixed Precision & Async Loading...

   Epoch [0/50] | Learning Rate: 0.000001

Running Evaluation — Epoch [0/50]...
Epoch [0/50] | Loss: 248.3160 | mAP@0.5: 0.0000 | Recall: 0.0000 | LR: 0.001000
  No improvement for 10 epochs

   Epoch [1/50] | Learning Rate: 0.000334

   Epoch [2/50] | Learning Rate: 0.000667

Running Evaluation — Epoch [2/50]...
Epoch [2/50] | Loss: 98.3734 | mAP@0.5: 0.6192 | Recall: 0.3866 | LR: 0.001000
  ★ New best model saved — mAP@0.5: 0.6192 at epoch 2

   Epoch [3/50] | Learning Rate: 0.000667

   Epoch [4/50] | Learning Rate: 0.000666

Running Evaluation — Epoch [4/50]...
Epoch [4/50] | Loss: 77.8741 | mAP@0.5: 0.7477 | Recall: 0.4667 | LR: 0.000664
  ★ New best model saved — mAP@0.5: 0.7477 at epoch 4

   Epoch [5/50] | Learning Rate: 0.000664

   Epoch [6/50] | Learning Rate: 0.000661

Running Evaluation — Epoch [6/50]...
Epoch [6/50] | Loss: 70.8855 | mAP@0.5: 0.7666 | Recall: 0.4805 | LR: 0.000656
  ★ New best mod

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch [49/50] | Loss: 25.0535 | mAP@0.5: 0.7505 | Recall: 0.5112 | LR: 0.000050
  No improvement for 180 epochs


Epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
Learning_Rate,███▆▆▅▅▅▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
Train_Loss,██▅▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Val_Recall,▁▁▆▇▇██████████████████████
Val_mAP_50,▁▁▆████████████████████████
Val_mAP_50_95,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Epoch,49
Learning_Rate,5e-05
Train_Loss,25.05348
Val_Recall,0.51121
Val_mAP_50,0.75055



Training Complete!
  Best : mobilevit_leyolo_best.pt  (epoch 14, mAP@0.5: 0.7996)
  Final: mobilevit_leyolo_final.pt
